In [29]:
#Libraries
import requests
import pandas as pd
import os

In [16]:
#Ticketmaster API KEY
API_KEY='LEIeBeElcqIf6xxGthrDmG1t5Ga7Weyq'

#Event Search
url = 'https://app.ticketmaster.com/discovery/v2/events.json'

In [4]:
#Top 10 cities in Canada (5) and USA (5):
cities = [("Toronto", "CA"),("Montreal", "CA"),("Vancouver", "CA"),("Calgary", "CA"),("Ottawa", "CA"),("New York", "US"),("Los Angeles", "US"),("Chicago", "US"),("Houston", "US"),("Miami", "US")]

In [24]:
#For Loop - To Collect Events

all_events =[]

for city, country in cities:
    parameters = {
        'apikey':API_KEY,
        'city':city,
        'countryCode':country,
        'size':50 # Number of events per city

    }
    try:
        response= requests.get(url,params=parameters)
        data = response.json()
        
        if '_embedded' in data:
            for event in data['_embedded']['events']:
                venue = event['_embedded']['venues'][0]
                price_info = event.get('priceRanges',[{}])[0]
                image_url = event['images'][0]['url'] if event.get('images') else None
                event_data = {
                    "event_title": event.get('name'),
                    "summary": event.get('info', ''),
                    "image_url": image_url,                   
                    "language": '',
                    "event_type": event.get('classifications', [{}])[0].get('segment', {}).get('name', ''),
                    "event_host": event.get('promoter', {}).get('name', ''),
                    "ticket_price": price_info.get('min'),
                    "booking_url": event.get('url'),
                    "start_date": event.get('dates', {}).get('start', {}).get('localDate'),
                    "end_date": '',
                    "start_time": event.get('dates', {}).get('start', {}).get('localTime'),
                    "end_time": '',
                    "event_place": venue.get('name'),
                    "full_address": venue.get('address', {}).get('line1', ''),
                    "country_name": venue.get('country', {}).get('name', ''),
                    "state_name": venue.get('state', {}).get('name', ''),
                    "city_name": venue.get('city', {}).get('name', ''),
                    "postal_code": venue.get('postalCode', '')
    
                }
                all_events.append(event_data)
    except Exception as e:
        print(f"Error when loading events for {city},{country}:{e}")


In [28]:
#Saving Data in CSV fomr
#Dataframe

csv_path = "ticketmaster_events.csv"
df =pd.DataFrame(all_events)

#CSV File
if os.path.exists(csv_path):
    df_existing = pd.read_csv(csv_path)
    df = pd.concat([df_existing, df], ignore_index=True)
    df.drop_duplicates(subset=["event_title", "start_date", "city_name"], inplace=True)

df.to_csv(csv_path, index=False)

df

,event_title,summary,image_url,language,event_type,event_host,ticket_price,booking_url,start_date,end_date,start_time,end_time,event_place,full_address,country_name,state_name,city_name,postal_code
0,Toronto Blue Jays vs. New York Yankees,NaN,https://s1.ticketm.net/dam/a/b7f/fa06b34f-1ba4...,NaN,Sports,PROMOTED BY VENUE,NaN,https://www.ticketmaster.ca/toronto-blue-jays-...,2025-06-30,NaN,19:07:00,NaN,Rogers Centre,1 Blue Jays Way,Canada,Ontario,Toronto,M5V 1J3
1,Toronto Blue Jays vs. New York Yankees,NaN,https://s1.ticketm.net/dam/a/b7f/fa06b34f-1ba4...,NaN,Sports,PROMOTED BY VENUE,NaN,https://www.ticketmaster.ca/toronto-blue-jays-...,2025-07-01,NaN,15:07:00,NaN,Rogers Centre,1 Blue Jays Way,Canada,Ontario,Toronto,M5V 1J3
2,Toronto Blue Jays vs. New York Yankees,NaN,https://s1.ticketm.net/dam/a/b7f/fa06b34f-1ba4...,NaN,Sports,PROMOTED BY VENUE,NaN,https://www.ticketmaster.ca/toronto-blue-jays-...,2025-07-02,NaN,19:07:00,NaN,Rogers Centre,1 Blue Jays Way,Canada,Ontario,Toronto,M5V 1J3
3,Toronto Blue Jays vs. New York Yankees,NaN,https://s1.ticketm.net/dam/a/b7f/fa06b34f-1ba4...,NaN,Sports,PROMOTED BY VENUE,NaN,https://www.ticketmaster.ca/toronto-blue-jays-...,2025-07-03,NaN,19:07:00,NaN,Rogers Centre,1 Blue Jays Way,Canada,Ontario,Toronto,M5V 1J3
4,Shania Twain,NaN,https://s1.ticketm.net/dam/a/d7c/734e9382-83e9...,NaN,Music,LIVE NATION CANADA (LN),NaN,https://www.ticketmaster.ca/shania-twain-toron...,2025-07-15,NaN,20:00:00,NaN,The Theatre at Great Canadian Casino Resort To...,1133 Queens Plate Dr,Canada,Ontario,Toronto,M9W0G4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
486,Playboi Carti,NaN,https://s1.ticketm.net/dam/a/9a5/6c2e42f8-634d...,NaN,Music,NaN,NaN,https://www.ticketmaster.com/playboi-carti-mia...,2026-01-06,NaN,19:30:00,NaN,Kaseya Center,601 Biscayne Blvd,United States Of America,Florida,Miami,33132
487,Kaseya Center All-Access Tour,NaN,https://s1.ticketm.net/dam/c/8cf/a6653880-7899...,NaN,NaN,NaN,NaN,https://www.universe.com/events/kaseya-center-...,2025-05-25,NaN,18:00:00,NaN,Kaseya Center,601 Biscayne Blvd,United States Of America,Florida,Miami,33132
488,Kaseya Center All-Access Tour,NaN,https://s1.ticketm.net/dam/c/8cf/a6653880-7899...,NaN,NaN,NaN,NaN,https://www.universe.com/events/kaseya-center-...,2025-05-26,NaN,13:00:00,NaN,Kaseya Center,601 Biscayne Blvd,United States Of America,Florida,Miami,33132
497,Guest Pass - HEAT vs. TBD,NaN,https://s1.ticketm.net/dam/c/8cf/a6653880-7899...,NaN,NaN,NaN,NaN,NaN,2025-05-26,NaN,19:00:00,NaN,General Admission,601 Biscayne Blvd,United States Of America,Florida,Miami,33132


In [30]:
#AI Help for storing images

import requests

In [31]:
image_folder = "C:/Users/yorbi/OneDrive - Lambton College/Term 3/Capstone/Project/Event_Images"
os.makedirs(image_folder, exist_ok=True)

for index, row in df.iterrows():
    image_url = row["image_url"]
    event_title = row["event_title"]

    if pd.notna(image_url): 
        try:
            # Clean file name to avoid invalid characteres 
            safe_title = "".join(c for c in event_title if c.isalnum() or c in (' ', '-', '_')).rstrip()
            filename = f"{safe_title[:40].replace(' ', '_')}_{index}.jpg"
            image_path = os.path.join(image_folder, filename)

            # Download images
            img_data = requests.get(image_url).content
            with open(image_path, 'wb') as handler:
                handler.write(img_data)

        except Exception as e:
            print(f"Downloading image error for {event_title}: {e}")